# Process OpenStreetMap Infrastructure Data

**Purpose.** Extract roads, railways, ports, aeroways, and waterways from annual Africa-wide OpenStreetMap PBF files.

**Inputs.** An Africa OSM PBF snapshot and the local output directory defined in the notebook.

**Outputs.** Infrastructure shapefiles separated by geometry type.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | Dependencies for OSM parsing and geospatial export
# !pip install osmium geopandas matplotlib shapely pandas

import osmium
import pandas as pd
import geopandas as gpd
from collections import defaultdict
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, Polygon



In [ ]:
# Cell | OSM snapshot path and tag census handler
Year = '23'

osm_file = "D:/Python/Africa_Data/africa-"+Year+"0101.osm.pbf"

class OSMStatsHandler(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.nodes = 0
        self.ways = 0
        self.relations = 0
        self.tags_count = defaultdict(int)
        self.infrastructure_tags = defaultdict(int)
        
        self.infra_keys = ['highway', 'railway', 'waterway', 'aeroway', 
                          'amenity', 'landuse', 'bridge', 
                          'tunnel', 'man_made']
    
    def node(self, n):
        self.nodes += 1
        for tag in n.tags:
            self.tags_count[tag.k] += 1
            if tag.k in self.infra_keys:
                self.infrastructure_tags[f"{tag.k}={tag.v}"] += 1
    
    def way(self, w):
        self.ways += 1
        for tag in w.tags:
            self.tags_count[tag.k] += 1
            if tag.k in self.infra_keys:
                self.infrastructure_tags[f"{tag.k}={tag.v}"] += 1
    
    def relation(self, r):
        self.relations += 1
        for tag in r.tags:
            self.tags_count[tag.k] += 1
            if tag.k in self.infra_keys:
                self.infrastructure_tags[f"{tag.k}={tag.v}"] += 1

print("OSM statistics handler defined")

In [ ]:
# Cell | Unified multimodal infrastructure extractor
# Extraction scope: roads, railways, ports, aeroways and waterways.
class UnifiedInfrastructureExtractor(osmium.SimpleHandler):
    """Extract selected infrastructure features in one pass."""
    
    def __init__(self):
        super().__init__()
        self.roads = []
        self.railways = []
        self.airports = []
        self.ports = []
        self.waterways = []
        
        self.road_types = ['motorway', 'trunk', 'primary', 'secondary', 'tertiary']
        
        self.railway_line_types = ['light_rail', 'miniature', 'monorail', 'narrow_gauge', 'rail', 'subway', 'tram']
        self.railway_polygon_types = ['platform', 'station']
        
        self.port_types = ['port', 'harbour']
        
        self.waterway_types = ['river', 'canal', 'fairway']
    
    def node(self, n):
        """Ignore node features."""
        pass
    
    def way(self, w):
        """Process line and polygon features."""
        tags_dict = dict(w.tags)
        coords = [(n.lon, n.lat) for n in w.nodes]
        
        if len(coords) < 2:
            return
        
        is_closed = coords[0] == coords[-1] and len(coords) >= 4
        geom = Polygon(coords) if is_closed else LineString(coords)
        
        if 'highway' in tags_dict and tags_dict['highway'] in self.road_types:
            if not is_closed:  # Apply the stated condition before continuing.
                self.roads.append({
                    'id': w.id,
                    'type': tags_dict['highway'],
                    'name': tags_dict.get('name', 'Unknown'),
                    'surface': tags_dict.get('surface', 'Unknown'),
                    'geometry': geom,
                    'tags': tags_dict
                })
        
        if 'railway' in tags_dict:
            railway_type = tags_dict['railway']
            if not is_closed and railway_type in self.railway_line_types:
                self.railways.append({
                    'id': w.id,
                    'type': railway_type,
                    'name': tags_dict.get('name', 'Unknown'),
                    'geometry': geom,
                    'tags': tags_dict
                })
            elif is_closed and railway_type in self.railway_polygon_types:
                self.railways.append({
                    'id': w.id,
                    'type': railway_type,
                    'name': tags_dict.get('name', 'Unknown'),
                    'geometry': geom,
                    'tags': tags_dict
                })
        
        if is_closed and 'public_transport' in tags_dict:
            pt_type = tags_dict['public_transport']
            if pt_type in self.railway_polygon_types:
                self.railways.append({
                    'id': w.id,
                    'type': pt_type,
                    'name': tags_dict.get('name', 'Unknown'),
                    'geometry': geom,
                    'tags': tags_dict
                })
        
        if 'waterway' in tags_dict:
            waterway_type = tags_dict['waterway']
            if not is_closed and waterway_type in self.waterway_types:
                self.waterways.append({
                    'id': w.id,
                    'type': waterway_type,
                    'name': tags_dict.get('name', 'Unknown'),
                    'geometry': geom,
                    'tags': tags_dict
                })
        
        if is_closed:
            if 'harbour' in tags_dict and tags_dict['harbour'] in self.port_types:
                self.ports.append({
                    'id': w.id,
                    'type': tags_dict['harbour'],
                    'name': tags_dict.get('name', 'Unknown'),
                    'geometry': geom,
                    'tags': tags_dict
                })
            elif 'seamark:type' in tags_dict and tags_dict['seamark:type'] in self.port_types:
                self.ports.append({
                    'id': w.id,
                    'type': tags_dict['seamark:type'],
                    'name': tags_dict.get('name', 'Unknown'),
                    'geometry': geom,
                    'tags': tags_dict
                })
            elif tags_dict.get('landuse') == 'port':
                self.ports.append({
                    'id': w.id,
                    'type': 'port',
                    'name': tags_dict.get('name', 'Unknown'),
                    'geometry': geom,
                    'tags': tags_dict
                })
        
        if 'aeroway' in tags_dict and is_closed:
            self.airports.append({
                'id': w.id,
                'type': tags_dict['aeroway'],
                'name': tags_dict.get('name', 'Unknown'),
                'geometry': geom,
                'tags': tags_dict
            })
    
    def relation(self, r):
        """Ignore relation features."""
        pass



In [ ]:
# Cell | Single-pass OSM parsing and GeoDataFrame creation
print("=" * 60)
print("=" * 60)

extractor = UnifiedInfrastructureExtractor()

# Parse the PBF snapshot once while retaining node locations for geometries.
extractor.apply_file(osm_file, locations=True)

print("\n" + "=" * 60)
print("=" * 60)

if extractor.roads:
    roads_gdf = gpd.GeoDataFrame(extractor.roads, crs='EPSG:4326')
    print(f"\n✓ Major roads: {len(roads_gdf):,} features")
    print(f"  Type distribution: {dict(roads_gdf['type'].value_counts())}")
else:
    print("\n✗ No major road data found")

if extractor.railways:
    railway_gdf = gpd.GeoDataFrame(extractor.railways, crs='EPSG:4326')
    print(f"\n✓ Railway network: {len(railway_gdf):,} features")
    print(f"  Main types: {', '.join(railway_gdf['type'].value_counts().head(5).index.tolist())}")
else:
    print("\n✗ No railway data found")

if extractor.airports:
    airport_gdf = gpd.GeoDataFrame(extractor.airports, crs='EPSG:4326')
    print(f"\n✓ Airport facilities: {len(airport_gdf):,} features")
    print(f"  Type: {airport_gdf['type'].unique()[0]}")
else:
    print("\n✗ No airport data found")

if extractor.ports:
    ports_gdf = gpd.GeoDataFrame(extractor.ports, crs='EPSG:4326')
    print(f"\n✓ Port facilities: {len(ports_gdf):,} features")
    named_ports = ports_gdf[ports_gdf['name'] != 'Unknown']
    if len(named_ports) > 0:
        print(f"  Named ports: {len(named_ports)} features")
        print(f"  Examples: {', '.join(named_ports['name'].head(5).tolist())}")
else:
    print("\n✗ No port data found")

if extractor.waterways:
    waterway_gdf = gpd.GeoDataFrame(extractor.waterways, crs='EPSG:4326')
    print(f"\n✓ Waterways: {len(waterway_gdf):,} features")
    print(f"  Main types: {', '.join(waterway_gdf['type'].value_counts().head(5).index.tolist())}")
else:
    print("\n✗ No waterway data found")

print("\n" + "=" * 60)
print("=" * 60)

In [ ]:
# Cell | Spatial coverage overview for extracted features
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

# Use a common map extent so feature layers are spatially comparable.
all_bounds = []
if 'roads_gdf' in locals() and len(roads_gdf) > 0:
    all_bounds.append(roads_gdf.total_bounds)
if 'railway_gdf' in locals() and len(railway_gdf) > 0:
    all_bounds.append(railway_gdf.total_bounds)
if 'airport_gdf' in locals() and len(airport_gdf) > 0:
    all_bounds.append(airport_gdf.total_bounds)
if 'ports_gdf' in locals() and len(ports_gdf) > 0:
    all_bounds.append(ports_gdf.total_bounds)
if 'waterway_gdf' in locals() and len(waterway_gdf) > 0:
    all_bounds.append(waterway_gdf.total_bounds)

if all_bounds:
    import numpy as np
    all_bounds = np.array(all_bounds)
    unified_bounds = [
        all_bounds[:, 0].min(),  # min x (longitude)
        all_bounds[:, 1].min(),  # min y (latitude)
        all_bounds[:, 2].max(),  # max x (longitude)
        all_bounds[:, 3].max()   # max y (latitude)
    ]
    x_margin = (unified_bounds[2] - unified_bounds[0]) * 0.05
    y_margin = (unified_bounds[3] - unified_bounds[1]) * 0.05
    unified_bounds = [
        unified_bounds[0] - x_margin,
        unified_bounds[1] - y_margin,
        unified_bounds[2] + x_margin,
        unified_bounds[3] + y_margin
    ]
else:
    unified_bounds = None

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

if 'roads_gdf' in locals() and len(roads_gdf) > 0:
    ax = axes[0, 0]
    roads_gdf.plot(ax=ax, linewidth=0.5, color='red', alpha=0.6)
    ax.set_title(f'Major Road Network (n={len(roads_gdf):,})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)
    if unified_bounds:
        ax.set_xlim(unified_bounds[0], unified_bounds[2])
        ax.set_ylim(unified_bounds[1], unified_bounds[3])
else:
    axes[0, 0].text(0.5, 0.5, 'No road data', ha='center', va='center', fontsize=12)
    axes[0, 0].set_title('Major Road Network')

if 'railway_gdf' in locals() and len(railway_gdf) > 0:
    ax = axes[0, 1]
    railway_gdf.plot(ax=ax, linewidth=0.8, color='blue', alpha=0.6)
    ax.set_title(f'Railway Network (n={len(railway_gdf):,})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)
    if unified_bounds:
        ax.set_xlim(unified_bounds[0], unified_bounds[2])
        ax.set_ylim(unified_bounds[1], unified_bounds[3])
else:
    axes[0, 1].text(0.5, 0.5, 'No railway data', ha='center', va='center', fontsize=12)
    axes[0, 1].set_title('Railway Network')

if 'airport_gdf' in locals() and len(airport_gdf) > 0:
    ax = axes[0, 2]
    airport_gdf.plot(ax=ax, color='purple', alpha=0.7)
    ax.set_title(f'Airport Facilities (n={len(airport_gdf):,})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)
    if unified_bounds:
        ax.set_xlim(unified_bounds[0], unified_bounds[2])
        ax.set_ylim(unified_bounds[1], unified_bounds[3])
else:
    axes[0, 2].text(0.5, 0.5, 'No airport data', ha='center', va='center', fontsize=12)
    axes[0, 2].set_title('Airport Facilities')

if 'ports_gdf' in locals() and len(ports_gdf) > 0:
    ax = axes[1, 0]
    ports_gdf.plot(ax=ax, color='navy', alpha=0.7)
    ax.set_title(f'Port Facilities (n={len(ports_gdf):,})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)
    if unified_bounds:
        ax.set_xlim(unified_bounds[0], unified_bounds[2])
        ax.set_ylim(unified_bounds[1], unified_bounds[3])
else:
    axes[1, 0].text(0.5, 0.5, 'No port data', ha='center', va='center', fontsize=12)
    axes[1, 0].set_title('Port Facilities')

if 'waterway_gdf' in locals() and len(waterway_gdf) > 0:
    ax = axes[1, 1]
    waterway_gdf.plot(ax=ax, linewidth=0.5, color='cyan', alpha=0.6)
    ax.set_title(f'Waterways (n={len(waterway_gdf):,})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)
    if unified_bounds:
        ax.set_xlim(unified_bounds[0], unified_bounds[2])
        ax.set_ylim(unified_bounds[1], unified_bounds[3])
else:
    axes[1, 1].text(0.5, 0.5, 'No waterway data', ha='center', va='center', fontsize=12)
    axes[1, 1].set_title('Waterways')

ax = axes[1, 2]
legend_handles = []

if 'roads_gdf' in locals() and len(roads_gdf) > 0:
    roads_gdf.plot(ax=ax, linewidth=0.3, color='red', alpha=0.4)
    legend_handles.append(mlines.Line2D([], [], color='red', linewidth=2, label='Roads'))

if 'railway_gdf' in locals() and len(railway_gdf) > 0:
    railway_gdf.plot(ax=ax, linewidth=0.5, color='blue', alpha=0.5)
    legend_handles.append(mlines.Line2D([], [], color='blue', linewidth=2, label='Railways'))

if 'waterway_gdf' in locals() and len(waterway_gdf) > 0:
    waterway_gdf.plot(ax=ax, linewidth=0.4, color='cyan', alpha=0.5)
    legend_handles.append(mlines.Line2D([], [], color='cyan', linewidth=2, label='Waterways'))

if 'ports_gdf' in locals() and len(ports_gdf) > 0:
    ports_gdf.plot(ax=ax, color='navy', alpha=0.8)
    legend_handles.append(mpatches.Patch(color='navy', label='Ports'))

if 'airport_gdf' in locals() and len(airport_gdf) > 0:
    airport_gdf.plot(ax=ax, color='purple', alpha=0.7)
    legend_handles.append(mpatches.Patch(color='purple', label='Airports'))

ax.set_title('Comprehensive Infrastructure Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)
if unified_bounds:
    ax.set_xlim(unified_bounds[0], unified_bounds[2])
    ax.set_ylim(unified_bounds[1], unified_bounds[3])

if legend_handles:
    ax.legend(handles=legend_handles, loc='best', fontsize=8)

plt.tight_layout()
plt.show()



In [ ]:
# Cell | Geometry-specific shapefile export
# Export results.
import os

output_dir = "E:/Africa_Infrastructure_Data/infrastructure/shp/20"+ str(int(Year)-1)
os.makedirs(output_dir, exist_ok=True)

saved_files = []

# Split features into point, line and polygon layers.
# Separate mixed geometries before Shapefile export.
def separate_by_geometry(gdf, base_name):
    """Return the geometry type."""
    results = {}
    
    points = gdf[gdf.geometry.type == 'Point']
    if len(points) > 0:
        results['Point'] = points
    
    lines = gdf[gdf.geometry.type.isin(['LineString', 'MultiLineString'])]
    if len(lines) > 0:
        results['LineString'] = lines
    
    polygons = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]
    if len(polygons) > 0:
        results['Polygon'] = polygons
    
    return results

if 'roads_gdf' in locals() and len(roads_gdf) > 0:
    geom_types = separate_by_geometry(roads_gdf, "africa_major_roads")
    
    for geom_type, gdf in geom_types.items():
        if geom_type == 'Point':
            output_file = os.path.join(output_dir, "africa_major_roads_point.shp")
        elif geom_type == 'LineString':
            output_file = os.path.join(output_dir, "africa_major_roads_line.shp")
        else:  # Polygon
            output_file = os.path.join(output_dir, "africa_major_roads_polygon.shp")
        
        export_gdf = gdf.copy()
        export_gdf['tags'] = export_gdf['tags'].apply(str)
        export_gdf.to_file(output_file, driver='ESRI Shapefile', encoding='utf-8')
        saved_files.append(f"Roads ({geom_type}): {output_file} ({len(gdf):,} features)")

if 'railway_gdf' in locals() and len(railway_gdf) > 0:
    geom_types = separate_by_geometry(railway_gdf, "africa_railways")
    
    for geom_type, gdf in geom_types.items():
        if geom_type == 'Point':
            output_file = os.path.join(output_dir, "africa_railways_point.shp")
        elif geom_type == 'LineString':
            output_file = os.path.join(output_dir, "africa_railways_line.shp")
        else:  # Polygon
            output_file = os.path.join(output_dir, "africa_railways_polygon.shp")
        
        export_gdf = gdf.copy()
        export_gdf['tags'] = export_gdf['tags'].apply(str)
        export_gdf.to_file(output_file, driver='ESRI Shapefile', encoding='utf-8')
        saved_files.append(f"Railways ({geom_type}): {output_file} ({len(gdf):,} features)")

if 'airport_gdf' in locals() and len(airport_gdf) > 0:
    geom_types = separate_by_geometry(airport_gdf, "africa_airports")
    
    for geom_type, gdf in geom_types.items():
        if geom_type == 'Point':
            output_file = os.path.join(output_dir, "africa_airports_point.shp")
        elif geom_type == 'LineString':
            output_file = os.path.join(output_dir, "africa_airports_line.shp")
        else:  # Polygon
            output_file = os.path.join(output_dir, "africa_airports_polygon.shp")
        
        export_gdf = gdf.copy()
        export_gdf['tags'] = export_gdf['tags'].apply(str)
        export_gdf.to_file(output_file, driver='ESRI Shapefile', encoding='utf-8')
        saved_files.append(f"Airports ({geom_type}): {output_file} ({len(gdf):,} features)")

if 'ports_gdf' in locals() and len(ports_gdf) > 0:
    geom_types = separate_by_geometry(ports_gdf, "africa_ports")
    
    for geom_type, gdf in geom_types.items():
        if geom_type == 'Point':
            output_file = os.path.join(output_dir, "africa_ports_point.shp")
        elif geom_type == 'LineString':
            output_file = os.path.join(output_dir, "africa_ports_line.shp")
        else:  # Polygon
            output_file = os.path.join(output_dir, "africa_ports_polygon.shp")
        
        export_gdf = gdf.copy()
        export_gdf['tags'] = export_gdf['tags'].apply(str)
        export_gdf.to_file(output_file, driver='ESRI Shapefile', encoding='utf-8')
        saved_files.append(f"Ports ({geom_type}): {output_file} ({len(gdf):,} features)")

if 'waterway_gdf' in locals() and len(waterway_gdf) > 0:
    geom_types = separate_by_geometry(waterway_gdf, "africa_waterways")
    
    for geom_type, gdf in geom_types.items():
        if geom_type == 'Point':
            output_file = os.path.join(output_dir, "africa_waterways_point.shp")
        elif geom_type == 'LineString':
            output_file = os.path.join(output_dir, "africa_waterways_line.shp")
        else:  # Polygon
            output_file = os.path.join(output_dir, "africa_waterways_polygon.shp")
        
        export_gdf = gdf.copy()
        export_gdf['tags'] = export_gdf['tags'].apply(str)
        export_gdf.to_file(output_file, driver='ESRI Shapefile', encoding='utf-8')
        saved_files.append(f"Waterways ({geom_type}): {output_file} ({len(gdf):,} features)")

for file_info in saved_files:
    print(f"✓ {file_info}")

if not saved_files:
    print("No data to export")
else:
    print(f"\nTotal: {len(saved_files)} shapefiles exported to: {output_dir}")
    print("\nGeometry types:")
    print("  - _point.shp: Point features")
    print("  - _line.shp: LineString/MultiLineString features")
    print("  - _polygon.shp: Polygon/MultiPolygon features")
    print("\nNote: Each shapefile includes .shp, .shx, .dbf, .prj and other associated files")
    print("You can now open these files directly in ArcMap!")